# Testing of ECGDataset for ECG-Hubert. 
This should generalize to the ECG-BigBird and ECG-PatchTST models as well

In [92]:
# add autoreload
%load_ext autoreload
%autoreload 2
import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb
import os
import sys
import re
import dotenv
from collections import defaultdict
from tqdm import tqdm

import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from transformers import pipeline
from transformers import AutoModel

from torch import float32

import torch

# presets for preprocessing: ECGHubert, ECGFounder Medxai

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from timex.ecg import dataset

INFO:root:Initializing Config class


In [3]:
dotenv.load_dotenv('../.env')
BASE_DIR = os.getenv('ECG_DIR')

In [4]:
PREPLIST = ['savgol', 'resampler', 'notch', 'bandpass', 'detrend', 'peak_trimming']

In [48]:
ecgConfig = dataset.Config
ecgConfig.SAMPLING_RATE = 500

ecgDS = dataset.ECGDataset(data=os.path.join(BASE_DIR, 'wilson-central-terminal-ecg-database-1.0.1'),
                           label_binarizer=None,
                           visualisation=False,
                           augmentations=[],
                           preprocessing=PREPLIST,
                           config=ecgConfig,
                           encode=True,
                           pretrain=False)

First 5 elements of the file_list: ['T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg01.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg02.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg03.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg04.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient002\\seg01.hea']
No labels available to show unique values


In [49]:
ecgDL = DataLoader(ecgDS, batch_size=32, shuffle=False, collate_fn=ecgDS.collate)

In [42]:
HubertECG = AutoModel.from_pretrained("Edoardo-BS/hubert-ecg-small", trust_remote_code=True,
                                        torch_dtype=float32, low_cpu_mem_usage=True)

In [43]:
HubertECG

HuBERTECG(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(4,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-2): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (3-4): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): HubertFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=512, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): HubertEncoder(
    (pos_conv_embed): HubertPositionalConvEmbedding(
      (conv): Parame

In [87]:
# use the DataLoader to extract the ECG signals
for batch in tqdm(ecgDL, desc="Processing ECG batches"):
    ecg_data, ecg_filenames = batch
    # process each ECG signal in the batch
    for signal, filename in zip(ecg_data, ecg_filenames):
        # Here you can apply the Hubert model to the signal
        # For example, you can use the model to extract features or perform classification
        features = HubertECG(signal[:12,:], 
                             attention_mask=None, 
                             output_attentions=False,
                             output_hidden_states=True, 
                             return_dict=True)  # Add batch dimension if needed
        print(f"Processed {filename}")
        break
    break


Processing ECG batches:   0%|          | 0/17 [00:04<?, ?it/s]

Processed T:\laupodteam\AIOS\Bram\data\ECG\wilson-central-terminal-ecg-database-1.0.1\patient001\seg01.hea


In [88]:
features['hidden_states'][1].shape

torch.Size([12, 77, 512])